# Week 8 - Linear Discriminant Analysis and Optimization

* FDA (2 classes)
    * Compute the class means (in PCA space)
    * Compute the within-class scatter matrix $\mathbf{S}_W$ and between-class scatter matrix $\mathbf{S}_B$
    * Compute the projection vector $\mathbf{w}$
    * Compute and plot the 1D projection of the data
    * Compute the class separation of the projected values
* LDA (3 classes)
    * Compute the within-class scatter matrix $\mathbf{S}_W$ and between-class scatter matrix $\mathbf{S}_B$
    * Compute the projection matrix $\mathbf{W}$
    * Compute and plot the 2D projection of the data
    * Compute the 2D LDA projection of the original 64D data 
* Constrained Optimization
    * Reformulate the problem as a maximization problem
    * Write out the Lagrangian function
    * Compute the gradients with respects to $x_1, x_2$, Lagrange multiplier $\lambda$ and the KKT multiplier $\mu$
    * Compute the optimum

In [ ]:
# Dependencies
import numpy as np
import scipy
import scipy.linalg
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA 
import matplotlib.pyplot as plt 
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns; sns.set_theme(); sns.set_palette('bright')

## Data
As usual, we'll be working with a dataset of handwritten digits. 
Let visualise some examples.

In [ ]:
# We're using a subset of two classes for now
digits = load_digits(n_class=2)

In [ ]:
# Handy plotting functions

def plot_examples():
    show_num = 4
    _, axes = plt.subplots(1, show_num)
    images_and_labels = list(zip(digits.images, digits.target))
    for ax, (image, label) in zip(axes[:], images_and_labels[:show_num]):
        ax.set_axis_off()
        ax.imshow(image, cmap=plt.cm.gray_r, interpolation='nearest')
        ax.set_title('Label: %i' % label)

def plot_scatter(data, target, alpha=0.5, legend=True,w=None):
    scatter = plt.scatter(data[:, 0], data[:, 1], c=target, edgecolor='none', alpha=alpha, cmap='rainbow')
    if legend:
        plt.legend(*scatter.legend_elements(), loc="upper right", title="Targets")
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')

    if w is not None:
        # w is a 2-D FDA direction vector (no bias term). The decision boundary
        # is the line perpendicular to w through the midpoint of the two class means.
        labels = np.unique(target)
        m1 = np.mean(data[target == labels[0]], axis=0)
        m2 = np.mean(data[target == labels[1]], axis=0)
        m_mid = (m1 + m2) / 2

        x_values = np.linspace(np.min(data[:, 0]), np.max(data[:, 0]), 100)
        # Perpendicular to w through m_mid: w[0]*(x - m_mid[0]) + w[1]*(y - m_mid[1]) = 0
        y_values = m_mid[1] - (w[0] / w[1]) * (x_values - m_mid[0])

        plt.plot(x_values, y_values, color='red', label='Decision boundary')
    #plt.show()
    
def plot_scatter3d(data, targets, view_point=(25, 45), alpha=0.5, legend=True):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(data[:,0], data[:,1], data[:,2], c=targets, cmap="rainbow", alpha=alpha)
    if legend:
        plt.legend(*scatter.legend_elements(), loc="upper right", title="Targets")
    ax.view_init(*view_point) # <- change viewpoint here
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.set_zlabel('Component 3')
    plt.show()

In [ ]:
plot_examples()

In [ ]:
# We can get a 2D version of the data using PCA
pca = PCA(n_components=2)
X = pca.fit_transform(digits.data) # this is the representation, we'll be working with
t = digits.target

In [ ]:
# Let's plot all the data in 2D
plot_scatter(X, t)

## 1) Fisher Discriminant Analysis (FDA)


### 1.1) Compute the class means (in PCA space)

In [ ]:
# X shape: (N, 2), t shape: (N,); classes are labelled 0 and 1
# TODO: compute the mean of each class -> m shape: (2, 2), where row m[k] (shape (2,)) is the mean of class k


### 1.2) Compute the within-class scatter matrix $\mathbf{S}_W$ and between-class scatter matrix $\mathbf{S}_B$
Use eq (4.27) and (4.28) from textbook for computing $\mathbf{S}_B$ and $\mathbf{S}_W$ respectively

In [ ]:
# X shape: (N, 2), t shape: (N,), m[k] shape: (2,)
# TODO: compute S_W (2x2) via eq. (4.28) and S_B (2x2) via eq. (4.27)
# Tip: reshaping a class mean to (1, 2) makes the outer products easy to write with @


### 1.3) Compute the projection vector $\mathbf{w}$
Hint: Use eq. (4.30) from textbook. You can use `np.linalg.pinv` for computing the pseudo-inverse. Also, remember to ensure that $||w||_2 = 1$

In [ ]:
# S_W shape: (2, 2), m[k] shape: (2,)
# TODO: compute the projection vector w (shape (2,)) via eq. (4.30), then normalise so ||w||_2 = 1


### 1.4) Compute and plot the 1D projection of the data
Hint: You can use `seaborn.displot` for a nice visualisation

In [ ]:
# w shape: (2,), X shape: (N, 2)
# TODO: project the data onto w -> X_proj shape: (N, 1); plot its 1D histogram per class


### 1.5) Compute the class separation of the projected values

In [ ]:
# w shape: (2,), m[k] shape: (2,)
# TODO: compute the class separation of the projected class means (a single scalar)


## 2) Linear Discriminant Analysis (LDA)

In [ ]:
# Data
n_class = 3
digits = load_digits(n_class=n_class)
pca = PCA(n_components=3)
X = pca.fit_transform(digits.data)
t = digits.target

plot_scatter3d(X, t, view_point=(35,35))

### 2.1) Compute the within-class scatter matrix $\mathbf{S}_W$ and between-class scatter matrix $\mathbf{S}_B$
See section 4.1.6 in the textbook or Lecture 15 (slide 17)

In [ ]:
# X shape: (N, 3), t shape: (N,) with 3 classes; m[k] shape: (3,), m_tot shape: (3,)
# TODO: compute S_W (3x3) and S_B (3x3) (see section 4.1.6)
# Tip: reshaping a class mean to (1, 3) makes the outer products easy to write with @


### 2.2) Compute the projection matrix $\mathbf{W}$
choose D'=2 (see lecture 15 slide 17 to understand D')

In [ ]:
# S_W shape: (3, 3), S_B shape: (3, 3)
# TODO: eigendecompose S_W^{-1} S_B, sort by descending eigenvalue, take the top D'=2 -> W shape: (3, 2)


### 2.3) Compute and plot the 2D projection of the data

In [ ]:
# W shape: (3, 2), X shape: (N, 3)
# TODO: project the data -> X_proj shape: (N, 2); plot it with plot_scatter


### 2.4) Compute and plot the 2D LDA projection of the original 64D data 

In [ ]:
# Data
n_class = 3
digits = load_digits(n_class=n_class)
X = digits.data
t = digits.target

In [ ]:
# Compute means

# compute within class and between-class scatter matrices

# Compute eigenvalues and eigenvectors, sort them and select top 2 eigenvectors

# perform projection

# plot projection in 2D


Comment on the general utility of the projection as compared to PCA

_Write your comment here._

_Note: Sections 1–2 above use material from Lecture 15 (FDA/LDA). The following section on constrained optimization requires Lecture 16 (Lagrangians and KKT conditions), the second lecture of Week 8._

# 3) Constrained Optimization
 
Consider the problem

minimize $f_{min}(x_1, x_2)$

subject to $ x_1 + x_2 \leq 4 
\quad \text{ and } \quad x_1 + 4x_2 = 5 
$
where $f_{min}(x_1, x_2) = (x_1 - 3)^2 + (x_2 - 2)^2$

## 3.1) Reformulate the problem as a canonical maximization problem
_Use the form described in the end of "Pattern Recognition and Machine Learning" Appendix E._

_NB: Appendix E writes the canonical **maximization** problem with inequality constraints in the form $h(x) \geq 0$ (with $\mu \geq 0$). This exercise's inequality is originally given as $x_1 + x_2 \leq 4$. To match Appendix E's form, define $h(x) = 4 - x_1 - x_2 \geq 0$ — a sign flip of the constraint, **not** an error in the book. Equivalently, if you prefer to keep $h(x) = x_1 + x_2 - 4 \leq 0$, you are using the standard convex-optimization convention from Lecture 16; just be consistent about the sign of $\mu$. Either choice gives the same optimum._

_Write your reformulation here._

## 3.2) Write out the Lagrangian function
_Use Equation (E.12) in "Pattern Recognition and Machine Learning" Appendix E., and write out all variables_

_Write your Lagrangian here._

## 3.3) Compute the gradients with respects to $x_1, x_2$, lagrange multiplier $\lambda$ and the KKT multiplier $\mu$

_Write your gradients here._

### 3.4) Compute the optimum
_Hint: Set it up as a system of linear equations and solve it using Gaussian Elimination (e.g. using `scipy.linalg.solve`)._

Here $\tilde{f} = -f_{min}$ is the maximization objective from step 3.1, and $h(x) = 4 - x_1 - x_2 \geq 0$.

Follow these steps:

1. **Assume the inequality constraint \( h(x) \) is inactive.**  
   - Set its KKT multiplier to zero (\( \mu = 0 \)).  
   - Include only the equality constraint \( g(x) = 0 \) in your formulation.  
   - Construct the system of equations from the KKT conditions:  
     \[
     \nabla \tilde{f}(x) + \lambda \nabla g(x) = 0, \quad g(x) = 0
     \]
   - Solve for \( x_1, x_2, \lambda \) using `scipy.linalg.solve`.

2. **Check the inequality constraint.**  
   - Evaluate \( h(x_1, x_2) \).  
   - If \( h(x_1, x_2) > 0 \), the constraint is inactive — keep this as your final solution.

3. **If the inequality is violated** (\( h(x_1, x_2) < 0 \)):  
   - Reformulate the problem assuming \( h(x) \) is **active** (\( h(x) = 0 \)).  
   - Include both constraints in the Lagrangian:  
     \[
     \mathcal{L}(x_1, x_2, \lambda, \mu) = \tilde{f}(x_1, x_2) + \lambda g(x_1, x_2) + \mu h(x_1, x_2)
     \]
   - Construct the corresponding system of equations:
     \[
     \nabla \tilde{f}(x) + \lambda \nabla g(x) + \mu \nabla h(x) = 0, \quad g(x) = 0, \quad h(x) = 0
     \]
   - Solve for \( x_1, x_2, \lambda, \mu \) using `scipy.linalg.solve`.

_The correct optimum is the solution that satisfies all constraints and yields the lowest \( f_{min}(x) \)._


In [ ]:
# Your code here